# Notebook 03 — Pipelines & Evaluation (Kaggle T4)

Runs all 3 systems and computes metrics (~2-3 hours)

In [ ]:
!pip install -q 'transformers>=4.45.0' accelerate bitsandbytes
!pip install -q colpali-engine byaldi open-clip-torch faiss-cpu
!pip install -q bert-score rouge-score nltk

In [ ]:
import os, sys, json, subprocess, importlib.util
import pandas as pd
from PIL import Image

WORKING_DIR = '/kaggle/working'
IMAGES_DIR = os.path.join(WORKING_DIR, 'openi', 'images')
COLPALI_INDEX_DIR = os.path.join(WORKING_DIR, 'colpali_index')
CLIP_INDEX_DIR = os.path.join(WORKING_DIR, 'clip_index')
HF_TOKEN = os.environ.get('HF_TOKEN')

# Clone repo if needed
REPO_PATH = os.path.join(WORKING_DIR, 'cxr-rag-system')
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/mohamedtaha77/cxr-rag-system.git', REPO_PATH], check=True)
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '-q'], check=True)

sys.path.insert(0, REPO_PATH)

print('✓ Setup complete')

In [ ]:
# Load corpus + test split
corpus_df = pd.read_csv(os.path.join(WORKING_DIR, 'reports_corpus.csv'))
test_df = corpus_df[corpus_df['split'] == 'test'].head(100).reset_index(drop=True)
study_to_impression = dict(zip(corpus_df['study_id'], corpus_df['impression']))

print(f'Evaluating on {len(test_df)} test studies')

In [ ]:
# Load modules
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

gen_mod = load_module('medgemma_generator', os.path.join(REPO_PATH, 'src', 'generation', 'medgemma_generator.py'))
colpali_mod = load_module('colpali_retriever', os.path.join(REPO_PATH, 'src', 'retrieval', 'colpali_retriever.py'))
clip_mod = load_module('clip_retriever', os.path.join(REPO_PATH, 'src', 'retrieval', 'clip_retriever.py'))
eval_mod = load_module('metrics', os.path.join(REPO_PATH, 'src', 'evaluation', 'metrics.py'))

MedGemmaGenerator = gen_mod.MedGemmaGenerator
ColPaliRetriever = colpali_mod.ColPaliRetriever
CLIPRetriever = clip_mod.CLIPRetriever
Evaluator = eval_mod.Evaluator

print('✓ Modules loaded')

In [ ]:
# Load MedGemma
generator = MedGemmaGenerator(hf_token=HF_TOKEN, load_in_4bit=True)
print('✓ MedGemma loaded')

In [ ]:
# Helper function
def run_report_pipeline(row, retriever, study_to_impression, k=3):
    image = Image.open(row['image_path']).convert('RGB')
    query = row.get('impression', 'chest x-ray findings')[:100]
    retrieved = retriever.search(query, k=k)
    context = [
        study_to_impression.get(
            os.path.splitext(os.path.basename(r.get('image_path', '')))[0], ''
        )
        for r in retrieved if r.get('image_path')
    ]
    context = [c for c in context if c][:k]
    return generator.generate_report(image, context_reports=context or None)

In [ ]:
# System A: ColPali + MedGemma
import torch, gc

print('Running System A (ColPali + MedGemma)...')
colpali = ColPaliRetriever.from_index(COLPALI_INDEX_DIR)
colpali.load_path_map(COLPALI_INDEX_DIR)

preds_A, refs_A = [], []
for i, (_, row) in enumerate(test_df.iterrows()):
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(test_df)}')
    pred = run_report_pipeline(row, colpali, study_to_impression)
    preds_A.append(pred)
    refs_A.append(row['impression'])

del colpali
gc.collect()
torch.cuda.empty_cache()
print('✓ System A done')

In [ ]:
# System B: CLIP + MedGemma
print('Running System B (CLIP + MedGemma)...')
clip = CLIPRetriever()
clip.load_index(CLIP_INDEX_DIR)

preds_B, refs_B = [], []
for i, (_, row) in enumerate(test_df.iterrows()):
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(test_df)}')
    pred = run_report_pipeline(row, clip, study_to_impression)
    preds_B.append(pred)
    refs_B.append(row['impression'])

del clip
gc.collect()
torch.cuda.empty_cache()
print('✓ System B done')

In [ ]:
# System C: MedGemma Direct
print('Running System C (MedGemma Direct)...')

preds_C, refs_C = [], []
for i, (_, row) in enumerate(test_df.iterrows()):
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(test_df)}')
    image = Image.open(row['image_path']).convert('RGB')
    pred = generator.generate_report(image, context_reports=None)
    preds_C.append(pred)
    refs_C.append(row['impression'])

print('✓ System C done')

In [ ]:
# Compute metrics
evaluator = Evaluator()

results = {}
for label, preds, refs in [
    ('ColPali + MedGemma', preds_A, refs_A),
    ('CLIP + MedGemma', preds_B, refs_B),
    ('MedGemma Direct', preds_C, refs_C),
]:
    print(f'Computing metrics for {label}...')
    results[label] = evaluator.evaluate_report_generation(preds, refs)

results_df = pd.DataFrame(results).T
print('\n=== Results ===')
print(results_df.round(4))

# Save
results_df.to_csv(os.path.join(WORKING_DIR, 'results.csv'))
print(f'\nSaved to {WORKING_DIR}/results.csv')

In [ ]:
# QA Evaluation
print('Running QA evaluation...')

qa_df = pd.read_json(os.path.join(WORKING_DIR, 'qa_dataset.jsonl'), lines=True)
qa_test = qa_df[qa_df['split'] == 'test'].head(50).reset_index(drop=True)

colpali = ColPaliRetriever.from_index(COLPALI_INDEX_DIR)
colpali.load_path_map(COLPALI_INDEX_DIR)

qa_preds, qa_refs = [], []
for i, (_, row) in enumerate(qa_test.iterrows()):
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(qa_test)}')
    image = Image.open(row['image_path']).convert('RGB')
    retrieved = colpali.search(row['question'], k=3)
    context = [
        study_to_impression.get(
            os.path.splitext(os.path.basename(r.get('image_path', '')))[0], ''
        )
        for r in retrieved if r.get('image_path')
    ]
    context = [c for c in context if c][:3]
    pred = generator.answer_question(image, row['question'], context)
    qa_preds.append(pred)
    qa_refs.append(row['answer'])

qa_metrics = evaluator.evaluate_qa(qa_preds, qa_refs)
print(f'\nQA Metrics: {qa_metrics}')

print('\n✓ Evaluation complete')